# Kinase Drug Discovery — Notebook 6
## Ligand-based modelling: proteochemometrics (PCM)

**Learning goals**

1. Build a **proteochemometric (PCM)** model that covers 12 kinases at once by combining compound fingerprints with a description of the **KLIFS binding pocket** (z-scales).
2. Compare PCM with per-kinase QSAR, a ligand-only model and a kinase-identity (one-hot) model, and test whether PCM can extrapolate to a kinase it has never seen (**leave-one-target-out**).
3. Identify which **pocket positions** the model uses.
4. Use the model to look at **kinase selectivity profiles**.

**Prerequisite:** run sections 1–2 of **Notebook 5**, which download and curate the ChEMBL data and write `data/kinase_dataset_curated.csv.gz`.

**Hardware / runtime** (4 CPU cores, no GPU, < 45 GB RAM)

| Step | Time (approx.) |
|---|---|
| Loading data and fingerprints | < 1 min |
| PCM random split (4 models) | 5–10 min |
| Leave-one-target-out | ~5–10 min per kinase |
| Full model, importances, selectivity profiles | 2–3 min |

Peak memory stays below ~4 GB. If you are short on time set `MAX_PER_TARGET` lower (e.g. 1500) in the configuration cell.

## 0. Setup and configuration

In [ ]:
import os
N_CPU = min(4, os.cpu_count() or 1)
for _v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(_v, str(N_CPU))

import time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import pearsonr, spearmanr

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(context="notebook", style="whitegrid")
pd.set_option("display.width", 160)

DATA = Path("data"); DATA.mkdir(exist_ok=True)
SEED = 42
rng = np.random.default_rng(SEED)
print("Using", N_CPU, "CPU threads")

In [ ]:
# ----------------------------- configuration -----------------------------
# 12 human kinases: ChEMBL single-protein target IDs (same panel and order as Notebook 5)
TARGETS = {
    "EGFR":  "CHEMBL203",  "ERBB2": "CHEMBL1824", "ABL1":  "CHEMBL1862", "SRC":   "CHEMBL267",
    "LCK":   "CHEMBL258",  "BTK":   "CHEMBL5251", "KIT":   "CHEMBL1936", "KDR":   "CHEMBL279",
    "FLT3":  "CHEMBL1974", "JAK2":  "CHEMBL2971", "AURKA": "CHEMBL4722", "CDK2":  "CHEMBL301",
}
ACTIVE_THRESHOLD = 6.5     # pChEMBL >= 6.5 (~316 nM) counts as "active" (Papyrus convention)
MAX_PER_TARGET = 3000      # subsample per kinase (None = use everything)
N_TREES = 200

## 1. Curated data and helper functions

We load the curated compound–kinase dataset produced in Notebook 5 (median pChEMBL per standardised compound–kinase pair) and redefine the helper functions used there.

In [ ]:
CURATED = DATA / "kinase_dataset_curated.csv.gz"
if not CURATED.exists():
    raise FileNotFoundError(f"{CURATED} not found — run sections 1–2 of Notebook 5 first "
                            "(they download and curate the ChEMBL data and write this file).")
agg = pd.read_csv(CURATED)
missing = [g for g in TARGETS if g not in set(agg.gene)]
if missing:
    print("WARNING: no curated data for", missing)

def standardize_smiles(smi):
    """Cleanup -> largest fragment -> neutralise -> canonical SMILES (None on failure). Same as Notebook 5."""
    try:
        m = Chem.MolFromSmiles(smi)
        if m is None:
            return None
        m = rdMolStandardize.Cleanup(m)
        m = rdMolStandardize.LargestFragmentChooser().choose(m)
        m = rdMolStandardize.Uncharger().uncharge(m)
        return Chem.MolToSmiles(m)
    except Exception:
        return None

def fingerprints(smiles, radius=2, n_bits=2048):
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    mols = [Chem.MolFromSmiles(s) for s in smiles]
    X = np.vstack([gen.GetFingerprintAsNumPy(m) for m in mols]).astype(np.uint8)
    bitvects = [gen.GetFingerprint(m) for m in mols]
    return X, bitvects

def regression_metrics(y_true, y_pred):
    return {"R2": r2_score(y_true, y_pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "Pearson r": pearsonr(y_true, y_pred)[0] if np.std(y_pred) > 0 else np.nan,
            "Spearman ρ": spearmanr(y_true, y_pred)[0] if np.std(y_pred) > 0 else np.nan,
            "n_test": len(y_true)}

agg.groupby("gene").agg(compounds=("smiles", "size"), median_pchembl=("pchembl", "median"),
                        frac_active=("active", "mean")).round(2).loc[[g for g in TARGETS if g in set(agg.gene)]]

## 2. Proteochemometrics (PCM)

A QSAR model (Notebook 5) describes **one** target. PCM models the **compound–target pair**:

$$\text{pActivity} = f(\text{compound descriptors},\ \text{protein descriptors})$$

so one model learns from *all* kinases simultaneously and can share information between related binding sites (van Westen *et al.*, Med Chem Commun 2011; Lapinsh *et al.*, 2001).

**Protein descriptors.** We use the 85 aligned KLIFS pocket residues (Notebook 1) and encode each residue with the 5 **z-scales** of Sandberg *et al.* (J Med Chem 1998) — principal components of 26 physicochemical properties (z1 ≈ lipophilicity, z2 ≈ size/polarisability, z3 ≈ polarity/charge, z4–z5 ≈ electronegativity, heat of formation, hardness). Gaps are encoded as zeros. Result: 85 × 5 = **425 protein descriptors**. Because the pocket is aligned, descriptor *j* means the same thing in every kinase.

In [ ]:
KLIFS_POCKETS = {
    "EGFR":  "KVLGSGAFGTVYKVAIKELEILDEAYVMASVDPHVCRLLGIQLITQLMPFGCLLDYVREYLEDRRLVHRDLAARNVLVITDFGLA",
    "ERBB2": "KVLGSGAFGTVYKVAIKVLEILDEAYVMAGVGPYVSRLLGIQLVTQLMPYGCLLDHVREYLEDVRLVHRDLAARNVLVITDFGLA",
    "ABL1":  "HKLGGGQYGEVYEVAVKTLEFLKEAAVMKEIKPNLVQLLGVYIITEFMTYGNLLDYLREYLEKKNFIHRDLAARNCLVVADFGLS",
    "SRC":   "VKLGQGCFGEVWMVAIKTLAFLQEAQVMKKLREKLVQLYAVYIVTEYMSKGSLLDFLKGYVERMNYVHRDLRAANILVVADFGLA",
    "LCK":   "ERLGAGQFGEVWMVAVKSLAFLAEANLMKQLQQRLVRLYAVYIITEYMENGSLVDFLKTFIEERNYIHRDLRAANILVIADFGLA",
    "BTK":   "KELGTGQFGVVKYVAIKMIEFIEEAKVMMNLSEKLVQLYGVFIITEYMANGCLLNYLREYLESKQFLHRDLAARNCLVVSDFGLS",
    "KIT":   "KTLGAGAFGKVVEVAVKMLALMSELKVLSYLGMNIVNLLGALVITEYCCYGDLLNFLRRFLASKNCIHRDLAARNILLICDFGLA",
    "KDR":   "KPLGRGAFGQVIEVAVKMLALMSELKILIHIGLNVVNLLGAMVIVEFCKFGNLSTYLRSFLASRKCIHRDLAARNILLICDFGLA",
    "FLT3":  "KVLGSGAFGKVMNVAVKMLALMSELKMMTQLGENIVNLLGAYLIFEYCCYGDLLNYLRSFLEFKSCVHRDLAARNVLVICDFGLA",
    "JAK2":  "QQLGKGNFGSVEMVAVKKLDFEREIEILKSLQDNIVKYKGVKLIMEYLPYGSLRDYLQKYLGTKRYIHRDLATRNILVIGDFGLT",
    "AURKA": "RPLGKGKFGNVYLLALKVLQLRREVEIQSHLRPNILRLYGYYLILEYAPLGTVYRELQKYCHSKRVIHRDIKPENLLLIADFGWS",
    "CDK2":  "EKIGEGTYGVVYKVALKKITAIREISLLKELNPNIVKLLDVYLVFEFLH-QDLKKFMDAFCHSHRVLHRDLKPQNLLILADFGLA",
}

# Sandberg et al. (1998) z-scales z1..z5
ZSCALES = {
    "A": [0.24, -2.32, 0.60, -0.14, 1.30],  "R": [3.52, 2.50, -3.50, 1.99, -0.17],
    "N": [3.05, 1.62, 1.04, -1.15, 1.61],   "D": [3.98, 0.93, 1.93, -2.46, 0.75],
    "C": [0.84, -1.67, 3.71, 0.18, -2.65],  "Q": [1.75, 0.50, -1.44, -1.34, 0.66],
    "E": [3.11, 0.26, -0.11, -3.04, -0.25], "G": [2.05, -4.06, 0.36, -0.82, -0.38],
    "H": [2.47, 1.95, 0.26, 3.90, 0.09],    "I": [-3.89, -1.73, -1.71, -0.84, 0.26],
    "L": [-4.28, -1.30, -1.49, -0.72, 0.84], "K": [2.29, 0.89, -2.49, 1.49, 0.31],
    "M": [-2.85, -0.22, 0.47, 1.94, -0.98], "F": [-4.22, 1.94, 1.06, 0.54, -0.62],
    "P": [-1.66, 0.27, 1.84, 0.70, 2.00],   "S": [2.39, -1.07, 1.15, -1.39, 0.67],
    "T": [0.75, -2.18, -1.12, -1.46, -0.40], "W": [-4.36, 3.94, 0.59, 3.44, -1.59],
    "Y": [-2.54, 2.44, 0.43, 0.04, -1.47],  "V": [-2.59, -2.64, -1.54, -0.85, -0.02],
    "-": [0.0, 0.0, 0.0, 0.0, 0.0],
}

def pocket_zscales(pocket):
    return np.array([ZSCALES.get(aa, ZSCALES["-"]) for aa in pocket], dtype=np.float32).ravel()

GENES = list(TARGETS)
PROT_Z = np.vstack([pocket_zscales(KLIFS_POCKETS[g]) for g in GENES])      # 12 x 425
PROT_ONEHOT = np.eye(len(GENES), dtype=np.float32)                            # 12 x 12 ("identity" descriptor)
print("z-scale protein descriptor matrix:", PROT_Z.shape)

In [ ]:
# PCM dataset: (optionally) subsample per kinase to keep training times short on 4 cores
parts = []
for g in GENES:
    sub = agg[agg.gene == g]
    if MAX_PER_TARGET and len(sub) > MAX_PER_TARGET:
        sub = sub.sample(MAX_PER_TARGET, random_state=SEED)
    parts.append(sub)
pcm = pd.concat(parts, ignore_index=True)
pcm["gene_idx"] = pcm.gene.map({g: i for i, g in enumerate(GENES)})

uniq_smiles = pcm.smiles.unique()
cmp_index = {s: i for i, s in enumerate(uniq_smiles)}
pcm["cmp_idx"] = pcm.smiles.map(cmp_index)
FP_PCM, FPS_PCM = fingerprints(uniq_smiles, radius=2, n_bits=1024)
print(f"PCM set: {len(pcm)} activities, {len(uniq_smiles)} compounds, {len(GENES)} kinases")

def build_X(rows, protein="zscales"):
    lig = FP_PCM[rows.cmp_idx.values].astype(np.float32)
    if protein is None:
        return lig
    P = PROT_Z if protein == "zscales" else PROT_ONEHOT
    return np.hstack([lig, P[rows.gene_idx.values]])

def make_rf_pcm():
    return RandomForestRegressor(n_estimators=N_TREES, max_features=0.2, min_samples_leaf=2,
                                 n_jobs=N_CPU, random_state=SEED)

### 2.1 Random split: does PCM beat single-target QSAR?

We compare four models on the same random 80/20 split (stratified by kinase):

| Model | Descriptors | Note |
|---|---|---|
| **QSAR (per kinase)** | fingerprint | 12 independent models |
| **Ligand-only (global)** | fingerprint | one model, ignores which kinase — predicts the same value for every kinase |
| **PCM one-hot** | fingerprint + kinase identity | protein descriptor carries no biology |
| **PCM z-scales** | fingerprint + KLIFS pocket z-scales | protein descriptor encodes physicochemistry of the pocket |

In [ ]:
train_idx, test_idx = train_test_split(np.arange(len(pcm)), test_size=0.2, random_state=SEED, stratify=pcm.gene)
p_tr, p_te = pcm.iloc[train_idx], pcm.iloc[test_idx]
preds = {}

t0 = time.time()
qsar_pred = np.zeros(len(p_te))
for g in GENES:
    mtr, mte = (p_tr.gene == g).values, (p_te.gene == g).values
    m = make_rf_pcm().fit(build_X(p_tr[mtr], None), p_tr.pchembl[mtr])
    qsar_pred[mte] = m.predict(build_X(p_te[mte], None))
preds["QSAR (per kinase)"] = qsar_pred
print(f"per-kinase QSAR done in {time.time() - t0:.0f}s")

for name, prot in [("Ligand-only (global)", None), ("PCM one-hot", "onehot"), ("PCM z-scales", "zscales")]:
    t0 = time.time()
    m = make_rf_pcm().fit(build_X(p_tr, prot), p_tr.pchembl)
    preds[name] = m.predict(build_X(p_te, prot))
    print(f"{name} done in {time.time() - t0:.0f}s")

In [ ]:
rows = []
for name, pr in preds.items():
    for g in GENES:
        mask = (p_te.gene == g).values
        r = regression_metrics(p_te.pchembl.values[mask], pr[mask]); r.update(model=name, gene=g)
        rows.append(r)
    r = regression_metrics(p_te.pchembl.values, pr); r.update(model=name, gene="ALL")
    rows.append(r)
res_random = pd.DataFrame(rows)

pivot = res_random.pivot(index="gene", columns="model", values="R2").loc[GENES + ["ALL"]]
display(pivot.round(3))
pivot.drop("ALL").plot.bar(figsize=(13, 4), width=0.8)
plt.ylabel("R² (test)"); plt.title("Random split: per-kinase performance"); plt.legend(loc="lower right")
plt.tight_layout(); plt.show()

**Interpretation.** With a random split, PCM and per-kinase QSAR usually perform similarly for data-rich kinases; PCM tends to help most for kinases with fewer data points because it borrows strength from related kinases. The ligand-only model is clearly worse: it cannot express that the same compound has different affinities for different kinases. One-hot and z-scale PCM are close here — when every kinase is present in training, an identity label is enough.

### 2.2 Leave-one-target-out (LOTO): predicting an unseen kinase

The real test for a protein descriptor is **extrapolation in target space**: train on 11 kinases, predict the 12th. Per-kinase QSAR is impossible here, and the one-hot descriptor of an unseen kinase is all zeros. Only the z-scale descriptor tells the model *how similar* the new pocket is to the known ones.

We run this with a kinase that has a close relative to the training set: KIT↔KDR/FLT3. But in total there are for four kinases that each have a close relative in the training set (ERBB2↔EGFR, LCK↔SRC, KIT↔KDR/FLT3, BTK↔other TKs). Takes ~5–10 minutes per kinase.

In [ ]:
LOTO_TARGETS = ["KIT"]
#"LCK", "BTK", "ERBB2", "KIT"
loto_rows = []
for g in LOTO_TARGETS:
    tr_rows, te_rows = pcm[pcm.gene != g], pcm[pcm.gene == g]
    for name, prot in [("Ligand-only (global)", None), ("PCM one-hot", "onehot"), ("PCM z-scales", "zscales")]:
        t0 = time.time()
        m = make_rf_pcm().fit(build_X(tr_rows, prot), tr_rows.pchembl)
        r = regression_metrics(te_rows.pchembl.values, m.predict(build_X(te_rows, prot)))
        r.update(left_out=g, model=name, seconds=round(time.time() - t0))
        loto_rows.append(r)
        print(f"{g:6s} {name:22s} R2={r['R2']:.3f}  rho={r['Spearman ρ']:.3f}  ({r['seconds']}s)")
res_loto = pd.DataFrame(loto_rows)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, metric in zip(axes, ["R2", "Spearman ρ"]):
    sns.barplot(data=res_loto, x="left_out", y=metric, hue="model", ax=ax)
    ax.set_title(f"Leave-one-target-out: {metric}"); ax.axhline(0, c="k", lw=0.8)
axes[1].get_legend().remove()
plt.tight_layout(); plt.show()

**Things to discuss**

* LOTO R² values can be low or even negative while Spearman ρ is clearly positive. Why? (Hint: the *average* potency level differs between kinases and data sets; ranking within a kinase can still be right.)
* For which left-out kinase does the z-scale PCM help most? Relate this to the pocket identity heat map from Notebook 1.
* Is the ligand-only model "cheating"? Many compounds are tested on several kinases, so its performance in LOTO partly reflects *promiscuity* of chemotypes.

**Exercise 6.1** — (a) Repeat LOTO for AURKA and CDK2 (serine/threonine kinases, few close relatives in the set). What happens? (b) Replace the z-scales by a **binary pocket one-hot encoding** (85 positions × 20 amino acids) and compare.

<details><summary><b>Solution sketch</b></summary>

```python
AA = "ACDEFGHIKLMNPQRSTVWY"
def pocket_onehot(p):
    v = np.zeros((85, 20), dtype=np.float32)
    for i, aa in enumerate(p):
        if aa in AA: v[i, AA.index(aa)] = 1
    return v.ravel()
PROT_Z_backup = PROT_Z.copy()
PROT_Z = np.vstack([pocket_onehot(KLIFS_POCKETS[g]) for g in GENES])   # reuse build_X(..., "zscales")
# ... rerun the LOTO loop with LOTO_TARGETS = ["AURKA", "CDK2"] ...
PROT_Z = PROT_Z_backup
```
For kinases without close relatives, extrapolation degrades sharply — PCM interpolates in target space, it does not magically extrapolate. Residue one-hot works but generalises less smoothly than physicochemical scales (cf. van Westen *et al.*, J Cheminform 2013, "Benchmarking of protein descriptor sets in proteochemometric modeling").
</details>

In [ ]:
# Your answer to Exercise 6.1 here

### 2.3 Which protein positions matter?

Random forests provide (impurity-based) feature importances. Summing the importances of the 5 z-scales per KLIFS position shows which pocket residues the PCM model uses to discriminate between kinases. We train on the full PCM set (≈1–2 min).

In [ ]:
t0 = time.time()
pcm_full = make_rf_pcm().fit(build_X(pcm, "zscales"), pcm.pchembl)
print(f"trained in {time.time() - t0:.0f}s")
imp = pcm_full.feature_importances_[FP_PCM.shape[1]:].reshape(85, 5).sum(axis=1)

KLIFS_REGIONS = [("I", 1, 3), ("g.l", 4, 9), ("II", 10, 13), ("III", 14, 19), ("αC", 20, 30),
                 ("b.l", 31, 37), ("IV", 38, 41), ("V", 42, 44), ("GK", 45, 45), ("hinge", 46, 48),
                 ("linker", 49, 52), ("αD", 53, 59), ("αE", 60, 64), ("VI", 65, 67), ("c.l", 68, 75),
                 ("VII", 76, 78), ("VIII", 79, 79), ("xDFG", 80, 83), ("a.l", 84, 85)]
fig, ax = plt.subplots(figsize=(16, 3.5))
ax.bar(range(1, 86), imp, color="steelblue")
for i, (name, s, e) in enumerate(KLIFS_REGIONS):
    ax.axvspan(s - 0.5, e + 0.5, color="grey" if i % 2 else "white", alpha=0.15)
    ax.text((s + e) / 2, imp.max() * 1.02, name, ha="center", fontsize=8, rotation=90, va="bottom")
ax.set_xlim(0.5, 85.5); ax.set_xlabel("KLIFS position"); ax.set_ylabel("importance")
ax.set_title("PCM: summed z-scale importance per pocket position", pad=35)
plt.tight_layout(); plt.show()
print("Top-10 positions:", [(int(p) + 1, KLIFS_POCKETS["EGFR"][p]) for p in np.argsort(-imp)[:10]])

Positions where all 12 kinases have the same residue (e.g. K17, E24, DFG) carry **zero** information — they are constant. Important positions are the variable ones lining the pocket, often including the gatekeeper (45), the residues around the hinge/solvent front and the back pocket. Caveat: impurity-based importances are biased towards features with many distinct values; permutation importance is more reliable but slower.

## 3. Kinase selectivity profiles

### 3.1 Measured profiles

Many compounds in ChEMBL have been tested on several of our 12 kinases. A selectivity matrix shows the polypharmacology that makes kinase inhibitors both effective and toxic.

In [ ]:
profile = agg.pivot_table(index="smiles", columns="gene", values="pchembl")[GENES]
multi = profile[profile.notna().sum(axis=1) >= 6]
print(f"{len(multi)} compounds measured on ≥ 6 of the 12 kinases")
top = multi.loc[multi.notna().sum(axis=1).sort_values(ascending=False).index[:40]]
fig, ax = plt.subplots(figsize=(10, 11))
sns.heatmap(top.reset_index(drop=True), cmap="rocket_r", vmin=4, vmax=10, ax=ax,
            cbar_kws={"label": "pChEMBL"}, mask=top.reset_index(drop=True).isna())
ax.set_facecolor("lightgrey"); ax.set_ylabel("compound"); ax.set_title("Measured profiles (grey = not measured)")
plt.tight_layout(); plt.show()

### 3.2 Predicted profiles of approved drugs

The PCM model can fill the gaps: it predicts every compound against every kinase in the panel. Below we predict a few approved inhibitors and compare with the measured median pChEMBL (where available).

> ⚠ These drugs are very likely **in the training data**, so this is a consistency check, not a validation. See Exercise 6.2.

In [ ]:
approved = {
    "imatinib":    "Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1",
    "dasatinib":   "Cc1nc(Nc2ncc(C(=O)Nc3c(C)cccc3Cl)s2)cc(N2CCN(CCO)CC2)n1",
    "nilotinib":   "Cc1cn(-c2cc(NC(=O)c3ccc(C)c(Nc4nccc(-c5cccnc5)n4)c3)cc(C(F)(F)F)c2)cn1",
    "gefitinib":   "COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1",
    "erlotinib":   "COCCOc1cc2ncnc(Nc3cccc(C#C)c3)c2cc1OCCOC",
    "lapatinib":   "CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5cccc(F)c5)c(Cl)c4)c3c2)o1",
    "ibrutinib":   "C=CC(=O)N1CCC[C@@H](n2nc(-c3ccc(Oc4ccccc4)cc3)c3c(N)ncnc32)C1",
    "ruxolitinib": "N#CC[C@H](C1CCCC1)n1cc(-c2ncnc3[nH]ccc23)cn1",
    "palbociclib": "CC(=O)c1c(C)c2cnc(Nc3ccc(N4CCNCC4)cn3)nc2n(C2CCCC2)c1=O",
}
appr_smiles = {k: standardize_smiles(v) for k, v in approved.items()}
gen1024 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
pred_profile = pd.DataFrame(index=list(approved), columns=GENES, dtype=float)
for name, smi in appr_smiles.items():
    fp = gen1024.GetFingerprintAsNumPy(Chem.MolFromSmiles(smi)).astype(np.float32)
    X = np.hstack([np.tile(fp, (len(GENES), 1)), PROT_Z])
    pred_profile.loc[name] = pcm_full.predict(X)
meas_profile = pd.DataFrame({name: profile.loc[smi] if smi in profile.index else pd.Series(np.nan, index=GENES)
                             for name, smi in appr_smiles.items()}).T[GENES]

fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), sharey=True)
sns.heatmap(pred_profile.astype(float), annot=True, fmt=".1f", cmap="rocket_r", vmin=4, vmax=10, ax=axes[0], cbar=False)
axes[0].set_title("PCM prediction")
sns.heatmap(meas_profile.astype(float), annot=True, fmt=".1f", cmap="rocket_r", vmin=4, vmax=10, ax=axes[1],
            mask=meas_profile.isna(), cbar_kws={"label": "pChEMBL"})
axes[1].set_facecolor("lightgrey"); axes[1].set_title("Measured (median ChEMBL; grey = no data in set)")
plt.tight_layout(); plt.show()

**Exercise 6.2** — Make the comparison honest: retrain the PCM model **without** any of the nine approved drugs *and* without their close analogues (Tanimoto ≥ 0.7), then predict their profiles again. Which profiles are still predicted correctly? Compute a simple selectivity score for each drug, e.g. the number of kinases with predicted pChEMBL ≥ 6.5.

<details><summary><b>Solution sketch</b></summary>

```python
appr_fps = [gen1024.GetFingerprint(Chem.MolFromSmiles(s)) for s in appr_smiles.values()]
max_sim = np.array([max(DataStructs.BulkTanimotoSimilarity(f, appr_fps)) for f in FPS_PCM])
keep_cmp = np.where(max_sim < 0.7)[0]
pcm_clean = pcm[pcm.cmp_idx.isin(keep_cmp)]
print(f"removed {len(pcm) - len(pcm_clean)} activities")
m_clean = make_rf_pcm().fit(build_X(pcm_clean, "zscales"), pcm_clean.pchembl)
# ... predict as above with m_clean ...
# selectivity: (pred_profile >= ACTIVE_THRESHOLD).sum(axis=1)
```
</details>

In [ ]:
# Your answer to Exercise 6.2 here

## 4. Summary and outlook

* **PCM** models compound–target pairs. With an informative protein descriptor (aligned pocket z-scales) a single model covers a kinase family and can make (careful) predictions for kinases with few or no data.
* With a random split PCM is comparable to per-kinase QSAR; the value of the protein descriptor only shows in **leave-one-target-out** validation, and only for kinases with close relatives in the training set — PCM interpolates in target space.
* Next steps in the literature: deep-learning PCM (e.g. graph neural networks + protein language-model embeddings), **multi-task** learning, uncertainty estimation via **conformal prediction**, and incorporating mutant data (e.g. EGFR T790M, ABL T315I) to predict resistance.

In **Notebook 3** we switch to the structure-based view: we dock the same inhibitors into ABL1 crystal structures and run a short molecular-dynamics simulation.